# 07 — History & Streaming

**Scope** — what it remembers.

When you finish you can:

- explain why an agent forgets everything between runs, and give it memory with `message_history=`
- tell `all_messages()` (the whole conversation) from `new_messages()` (just this run)
- save a conversation as JSON and load it back with `ModelMessagesTypeAdapter`
- stream an answer as it is written, with the course's one copy-paste `async with` pattern

In [ ]:
from learningdantic import setup

MODEL = setup()

## An agent remembers only what you hand it

Every `agent.run()` starts from zero. The model forgets you the moment the run ends.
Memory is a list of messages that **you** pass back in with `message_history=`.

That makes memory part of the agent's **scope**: it can remember exactly what is in that list,
and nothing else. `result.all_messages()` gives you the whole conversation so far, ready to pass on.

In [ ]:
from pydantic_ai import Agent

planner = Agent(
    MODEL,
    instructions="You are a laconic travel planner. Keep answers under 25 words unless asked otherwise.",
)

first = await planner.run("I have one day in Paris. Pick one theme for the day.")
print("turn 1    :", first.output)

second = await planner.run(
    "Turn that theme into a 3-stop itinerary.",
    message_history=first.all_messages(),  # hand turn 1 back in
)
print("turn 2    :", second.output)

stranger = await planner.run("Which theme did we pick?")  # no history at all
print("no history:", stranger.output)

> ✅ **Checkpoint** — turn 2 builds on turn 1's theme without you repeating it. The last run got
> no `message_history`, so the model has no idea what "we picked" (it asks, or admits it doesn't
> know). Same agent; only the list changed.

## Keep one `history` variable

In a real chat you keep one list and replace it after every turn. The result has two methods:

- `result.all_messages()` — the **whole** conversation: what you passed in, plus this run.
- `result.new_messages()` — **only this run's** messages: what a server would add to a database.

Start from the latest result (`second`), so the itinerary is part of what the model sees.

In [ ]:
history = second.all_messages()  # turns 1 and 2, including the itinerary
for ask in ["Swap the last stop for something cheaper.", "Now sum up the plan in one line."]:
    r = await planner.run(ask, message_history=history)
    history = r.all_messages()  # replace the list after every turn
    print("→", r.output)

print("all messages:", len(r.all_messages()), "(the whole conversation)")
print("new messages:", len(r.new_messages()), "(just this run)")

> ✅ **Checkpoint** — the swap changes the last stop of *your* itinerary, because it was in the
> history. Then `all messages: 8` and `new messages: 2`: each turn adds one request and one
> response, and this was the 4th turn.

## Save a conversation and load it back

`ModelMessagesTypeAdapter` turns a message list into JSON bytes and back. That is how a chat is
kept in a file or a database between sessions. The messages hold everything the model saw,
including instructions and tool results, so treat stored history as user data.

Aside: Pydantic AI 2.x also has `result.conversation`, passed back with `agent.run(..., conversation=...)`;
it adds running usage and a conversation id. `message_history` is all this course needs.
See the [message history docs](https://pydantic.dev/docs/ai/core-concepts/message-history).

In [ ]:
from pydantic_ai import ModelMessagesTypeAdapter

blob = ModelMessagesTypeAdapter.dump_json(history)  # messages -> JSON bytes
print("saved", len(blob), "bytes of JSON")

restored = ModelMessagesTypeAdapter.validate_json(blob)  # JSON bytes -> messages
r = await planner.run("Which theme did I pick at the very start?", message_history=restored)
print(r.output)

> ✅ **Checkpoint** — a byte count, then the planner names the theme from turn 1. It only had
> the copy that went through JSON, so saving and loading works.

## Streaming: watch the answer arrive

`run_stream` hands you the answer piece by piece while the model writes it, so a long answer
starts showing right away. This is the course's one copy-paste async pattern. Two new Python words:

- `async with ... as stream:` opens something and closes it when the block ends. Here, the stream.
- `async for piece in ...:` is a `for` loop over pieces that arrive over time.

`stream_text(delta=True)` yields only the **new** text each time, so printing every piece prints
the poem once. After the loop, `await stream.get_output()` gives the finished answer.
This pattern is for text answers: `stream_text` raises an error if the agent has an `output_type`.

In [ ]:
pieces = []
async with planner.run_stream("Write a 4-line poem about debugging at 2am.") as stream:
    async for delta in stream.stream_text(delta=True):
        pieces.append(delta)
        # end="": stay on the same line; flush=True: show it right away
        print(delta, end="", flush=True)
    output = await stream.get_output()
print()
print("---")
print(len(pieces), "pieces; joined they match the final output:", "".join(pieces) == output)

> ✅ **Checkpoint** — the poem appears a bit at a time, once. Then a line like
> `3 pieces; joined they match the final output: True` (the count varies: a fast model sends bigger pieces).
>
> For chat UIs there is also `agent.run_stream_events(...)`, used the same way:
> `async with agent.run_stream_events(...) as events:` then `async for event in events:`.
> It yields every event, tool calls included. Notebook 13's `agent.to_web()` uses it for you.

## 🏋️ Exercise — a Chat class

Wrap memory in a small class. `ask` runs the agent, so it follows the async rule: write
`async def ask(...)`, and `await chat.ask(...)` when you call it. `save` and `load` only touch
files, so they are plain `def`. (`@classmethod` makes a method you call on the class itself,
like `Chat.load(...)`; here it builds a new `Chat`.)

```python
class Chat:
    def __init__(self, agent): ...
    async def ask(self, prompt: str) -> str: ...  # run with self.history, then update it
    def save(self, path): ...                      # ModelMessagesTypeAdapter -> file
    @classmethod
    def load(cls, agent, path): ...                # file -> a new Chat with that history
```

1. Build `Chat` and hold a 3-turn conversation with one agent.
2. `save` it to `../data/07_chat.json` (run `Path("../data").mkdir(exist_ok=True)` first),
   `load` it into a **new** `Chat`, and show that the new object remembers turn 1.

**Stretch:** add `async def ask_stream(self, prompt)` that prints the answer as it streams
(the pattern above) and sets `self.history = stream.all_messages()` at the end.

Your take: [`../solutions/07_chat_class.ipynb`](../solutions/07_chat_class.ipynb).

## Next →

**[08_offline_testing.ipynb](08_offline_testing.ipynb)** — everything so far cost tokens to check.
Next you test agents offline, for free.